# 🎓 Examen Final — Pipeline ELT Complet 

**Contexte métier :**
Vous avez été recruté(e) en tant que Data Engineer pour *TheLook eCommerce*.
Votre mission d'aujourd'hui est de construire le pipeline de données quotidien des produits du catalogue.
Ce pipeline extraira les données depuis un fichier stocké dans le cloud (qui remplace une API), les nettoiera, puis calculera des indicateurs clés (KPIs), le tout automatisé via GitHub Actions.

---

### 📋 Règles de l'examen

- Le travail s'effectue individuellement.
- Vous devez construire une architecture **Médaillon (Bronze, Silver, Gold)**.
- Toutes les tables BigQuery doivent être créées dans votre dataset personnel : `examen_<votre_prenom>`. 
- Sauvez vos requetes sur BigQuery
- Le rendu final sera un dépôt GitHub contenant votre code.

---

## 🛠️ Outils à disposition

| Couche | Outil | Action |
|:---|:---|:---|
| **Source ➔ Bronze** | **Python (dlt)** | Ingestion des données brutes avec `pipeline.run()` |
| **Bronze ➔ Silver** | **SQL (BigQuery)** | Nettoyage, typage (`SAFE_CAST`, `COALESCE`, `NULLIF`) |
| **Silver ➔ Gold** | **SQL (BigQuery)** | Agrégation (`GROUP BY`, `COUNT`, `AVG`) |
| **Orchestration** | **GitHub Actions** | Automatisation CI/CD (Workflow YAML) |

---
## 📦 Étape 1 : Ingestion des données (Couche Bronze) 

Le fichier source est un fichier JSON volontairement contenant des erreurs de typage et des données manquantes.
Il est accessible via la console ou le cloud shell.

**Votre mission :**
1. Sur votre machine locale, créez un fichier `load_products.py`.
2. Complétez le script Python ci-dessous (le "Starter Kit") pour ingérer ce fichier JSON dans une table `bronze_products` dans BigQuery.



### Configuration de l'authentification Google Cloud
Pour que `dlt` puisse lire sur Cloud Storage et écrire sur BigQuery, il doit s'authentifier. **Deux options s'offrent à vous :**

#### Option 1 : Via Google Cloud SDK (`gcloud`) - La vraie méthode
1. **Installez le SDK Google Cloud** sur votre machine (Windows, Mac ou Linux) en suivant ce lien : [Installer gcloud SDK](https://cloud.google.com/sdk/docs/install-sdk?hl=fr)
2. Dans votre terminal, exécutez la commande suivante :
   ```bash
   gcloud auth application-default login
   ```
3. Connectez-vous sur la page web qui s'ouvre.
4. *Note : Si vous choisissez cette option, vous pouvez ignorer les lignes `os.environ[...]` dans la cellule ci-dessous.*

#### Option 2 : Via la clé JSON (Solution de secours)
1. Récupérez le fichier `cle-examen.json` fourni par le professeur.
2. Placez-le **exactement** dans le même dossier que ce notebook.
3. Laissez le code `os.environ[...]` actif dans la cellule ci-dessous.


In [ ]:
# Fichier: ingestion/load_products.py
# --- Starter Kit à copier dans votre projet ---

import dlt
import json
from dlt.sources.filesystem import filesystem
import os



# 1. Création d'une ressource filesystem pointant vers le bucket
# L'authentification GCP est gérée automatiquement par WIF (GitHub Actions) ou `gcloud auth` (Local)
gcs_source = filesystem(
    bucket_url="gs://exam_bucket_eu/",
    file_glob="**/*.json"
)

# 2. Définition d'un lecteur (transformer) JSON personnalisé
@dlt.transformer
def read_json(file_items):
    for item in file_items:
        with item.open("r", encoding="utf-8") as f:
            yield from json.load(f)

# 3. Initialisation du pipeline dlt
pipeline = dlt.pipeline(
    pipeline_name="exam_pipeline",
    destination="bigquery",         
    dataset_name="examen_lucas"         
)

# 4. Exécution du chargement en combinant la ressource et le lecteur JSON
load_info = pipeline.run(
    gcs_source | read_json, 
    table_name="bronze_products",     
    write_disposition="replace" 
)

print("✅ Chargement terminé !")
print(load_info)


###  Vérification locale avec pandas

Maintenant que le pipeline a tourné, utilisez les fonctionnalités natives de `dlt` pour interroger directement votre dataset BigQuery et récupérer les données de la table `bronze_products` sous forme de **DataFrame pandas**, afin de vérifier visuellement le résultat de votre ingestion depuis ce notebook.

In [1]:
# Écrivez ici le code dlt pour charger la table bronze dans un DataFrame
import dlt

pipeline = dlt.attach(pipeline_name="exam_pipeline")
df = pipeline.dataset().bronze_products.df()
df.head()

/Users/lucasporcheron/Documents/Cours/OGEC/M2/BigData/cours_lasalle/big_data/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


,product_id,name,category,cost,retail_price,is_active,added_date,_dlt_load_id,_dlt_id
0,13010,Urban Sleep & Lounge 3010,NaN,19.8,46.10,True,2023-01-01,1791547399.804078,b3jIxpSG1AZ/Xw
1,16432,Casual Underwear 6432,NaN,37.2,78.32,False,2023-01-01,1791547399.804078,2B/yTgvQHrmtXQ
2,22277,Classic Maternity 12277,NaN,10.22,23.00,True,2023-01-01,1791547399.804078,qcFqmTnqJnO0kg
3,10094,Sporty Suits & Sport Coats 94,NaN,22.86,62.00,True,2023-01-02,1791547399.804078,4DxinhUJp2fUkw
4,10492,Urban Leggings 492,NaN,49.2,91.00,False,2023-01-02,1791547399.804078,lDFjoJN/NRb3kA


In [ ]:
# Récupérez ici des informations pertinentes sur l'execution du pipeline dlt
print(pipeline.last_trace)    # résumé complet de l'execution du pipeline                     


Run started at 2026-10-09 12:03:17.991065+00:00 and COMPLETED in 15.54 seconds with 4 steps.
Step extract COMPLETED in 2.22 seconds.

Load package 1791547399.804078 is EXTRACTED and NOT YET LOADED to the destination and contains no failed jobs

Step normalize COMPLETED in 0.40 seconds.
Normalized data for the following tables:
- bronze_products: 30000 row(s)

Load package 1791547399.804078 is NORMALIZED and NOT YET LOADED to the destination and contains no failed jobs

Step load COMPLETED in 11.13 seconds.
Pipeline exam_pipeline load step finished in 10.81 seconds
1 load package(s) were loaded to destination bigquery and into dataset examen_lucas
The bigquery destination used None@lasalle-big-data location to store data
Load package 1791547399.804078 is LOADED and contains no failed jobs

Step run COMPLETED in 15.54 seconds.
Pipeline exam_pipeline load step finished in 10.81 seconds
1 load package(s) were loaded to destination bigquery and into dataset examen_lucas
The bigquery destina

---
## 🧹 Étape 2 : Nettoyage des données (Couche Silver)

> IMPORTANT: créer un Fichier: transformations/silver.sql

Félicitations, vos données sont dans BigQuery ! Mais elles sont très "sales" (Bronze).
Regardez le schéma de votre table `bronze_products` dans la console BigQuery

**Votre mission :**
Écrivez une requête SQL (sauvegardée dans un fichier `transformations/silver.sql`) qui lit la table `bronze_products` et crée (ou remplace) une table `silver_products` totalement propre.

> 💡 **Aide-mémoire SQL & Astuce dlt :**
> - **Le piège dlt (Variant Columns)** : si dlt détecte un changement de type (entier puis décimal), il crée deux colonnes distinctes (ex: `retail_price` et `retail_price__v_double`).
> - **Fusionner deux colonnes** : La fonction `COALESCE()` retourne la première valeur non-nulle. 
>
>   *Exemple (fusionner 2 numéros de téléphone)* : `COALESCE(telephone_portable, CAST(telephone_fixe AS STRING)) AS telephone`
> - Pour changer le type d'une colonne de manière sécurisée : `SAFE_CAST(colonne AS FLOAT64)`
> - Pour transformer un mot exact en `null` : `NULLIF(colonne, 'NULL')`
> - Pour la création de la table : `CREATE OR REPLACE TABLE \`votre_projet.votre_dataset.silver_products\` AS SELECT ...`


---
## 📈 Étape 3 : Agrégation Métier (Couche Gold) 

> IMPORTANT: créer un Fichier: transformations/gold.sql

Maintenant que vous avez une belle table `silver_products`, nous voulons extraire de la valeur métier.

**Votre mission :**
Écrivez une requête SQL (sauvegardée dans `transformations/gold.sql`) qui crée (ou remplace) une table `gold_category_metrics`.

**Résultat attendu :**
Pour chaque **catégorie** de vêtement, nous voulons connaître :
1. Le nombre total de produits.
2. Le prix de vente moyen.
3. La marge unitaire moyenne (calcul de la marge = prix de vente - coûts ).

Triez le résultat pour afficher les catégories les plus rentables (marge moyenne la plus élevée) en premier.

---
## ⚙️ Étape 4 : Orchestration (CI/CD GitHub Actions)

Vous avez 3 scripts (`load_products.py`, `silver.sql`, `gold.sql`). En production, on ne va pas les lancer à la main tous les jours ! Nous allons utiliser GitHub Actions pour orchestrer l'ordre d'exécution (le pipeline ELT).

**Votre mission :**
1. Sur votre dépôt GitHub, créez le dossier `.github/workflows/` et ajoutez-y le fichier `elt_pipeline.yml` ci-dessous.
2. Décommentez et complétez les 3 étapes (Jobs) à la fin du fichier.
3. *Le professeur a déjà configuré une Workload Identity Federation (WIF) dans Google Cloud. Vous n'avez pas besoin de clé JSON !* Suivez simplement les instructions d'authentification ci-dessous.

> 💡 **Exécution des requêtes SQL depuis le terminal :** 
> `bq query --use_legacy_sql=false < transformations/votre_fichier.sql`

In [ ]:
# Fichier: .github/workflows/elt_pipeline.yml
name: Pipeline ELT Quotidien

# Le pipeline se lance manuellement, ou tous les jours à minuit
on:
  workflow_dispatch:
  schedule:
    - cron: 

jobs:
  run-elt:
    runs-on: ubuntu-latest
    
    # Permissions requises pour utiliser Workload Identity Federation (OIDC) avec GitHub Actions
    permissions:
      contents: 'read'
      id-token: 'write'

    steps:
      - name: Récupérer le code (Checkout)
        uses: actions/checkout@v4

      - name: Authentification Google Cloud (WIF)
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: 'projects/833312929773/locations/global/workloadIdentityPools/github-pool/providers/github-provider'
          service_account: 'github-actions-elt@lasalle-big-data.iam.gserviceaccount.com'

      - name: Installation de Python et dlt
        uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: |
          uv add "dlt[bigquery,gs]"         

      # --- 🟢 ÉTAPE BRONZE : INGESTION ---
      # - name: 1. Ingestion API vers Bronze
      #   run: 

      # --- ⚪ ÉTAPE SILVER : TRANSFORMATION ---
      # - name: 2. ransformation Bronze vers Silver
      #   run: # À COMPLÉTER (Utiliser bq query)

      # --- 🟡 ÉTAPE GOLD : AGRÉGATION ---
      # - name: 3. Agrégation Silver vers Gold
      #   run: # À COMPLÉTER


---
### ✅ Livrable attendu

Sur votre repo GitHub, la structure de fichiers doit ressembler exactement à ceci :

```
mon_projet/
│__ce_notebook.ipynb
|
├── .github/
│   └── workflows/
│       └── elt_pipeline.yml
│
├── ingestion/
│   └── load_products.py
│
└── transformations/
    ├── silver.sql
    └── gold.sql
```

_pensez a sauver vos requetes SQL dans BiGQuery ! Ex: examen_prenom_requete_qui_fait_quelquechose_

**Bon courage !**